# Extending Fluopy with custom fluorophores and transitions

Fluopy provides built-in fluorophore data, photophysical states, and transition
types. This tutorial shows how to define additional objects without modifying
Fluopy's source code.

## Main objects

| Object | Purpose |
|---|---|
| Spectrum | Wavelength-dependent emission or absorption data |
| FluorophoreData | Spectra and photophysical constants of one fluorophore type |
| Fluorophore | A fluorophore type, position, and corresponding data |
| SingleState | A state occupied by one fluorophore |
| PairedState | Donor and acceptor states involved in an energy-transfer transition |
| TransitionType | Initial state, final state, abbreviation, and photon emission |
| Transition | A transition type combined with a rate and fluorophore identities |
| TransitionSet | A collection of transitions ready for simulation or prediction |

## Define a custom fluorophore from arrays

Spectra can be constructed directly from wavelength and value arrays.

The S0 absorption spectrum is required to derive excitation. Its values must be
absolute molar extinction coefficients. An emission spectrum is required for
bandpass filtering and for using the fluorophore as an energy-transfer donor.
Emission intensities may be relative.

After defining the spectra:

1. Create a FluorophoreData object containing the spectra and constants.
2. Create a Fluorophore using that data.
3. Create a FluorophoreSystem.
4. Call FluorophoreSystem.load_transitions() to derive built-in transitions.

In [ ]:
import numpy as np

import fluopy

# creating the spectra from arrays
wavelengths = np.array([600, 620, 640, 660, 680])

emission = fluopy.Spectrum.from_arrays(
    wavelengths=wavelengths,
    values=[0.0, 0.2, 1.0, 0.6, 0.1],
)
absorption_s0 = fluopy.Spectrum.from_arrays(
    wavelengths=wavelengths,
    values=[10000, 40000, 80000, 30000, 5000],
)

# creating the FluorophoreData object
custom_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=emission,
    absorption_spectra={"s0": absorption_s0},
)

# creating the Fluorophore
custom_fluorophore = fluopy.Fluorophore(
    name="custom",
    position=[0, 0],
    constants=custom_data,
)

# creating the FluorophoreSystem
fluorophore_system = fluopy.FluorophoreSystem(
    fluorophores=[custom_fluorophore],
)

# loading the built-in transitions
transitions = fluorophore_system.load_transitions(
    wavelength=640,
    energy_transfer=False,
    dstorm=False,
)
transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Load custom spectra from CSV files

Spectrum.from_csv() reads wavelength-dependent data from a CSV file. By default,
the wavelength and spectrum columns are named Wavelengths and y. Alternative
column names can be specified with wavelength_column and value_column.

The example uses spectrum files bundled with Fluopy so that it can be executed
as part of the documentation build. Replace these paths with paths to external
CSV files when using custom experimental data.

In [ ]:
from pathlib import Path

import fluopy

spectrum_dir = Path(fluopy.__file__).parent / "fluorophore_spectra" / "atto643_data"

custom_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=fluopy.Spectrum.from_csv(spectrum_dir / "emission.csv"),
    absorption_spectra={
        "s0": fluopy.Spectrum.from_csv(spectrum_dir / "absorption_s0.csv"),
    },
)

custom_fluorophore = fluopy.Fluorophore(
    name="custom",
    position=[0, 0],
    constants=custom_data,
)
fluorophore_system = fluopy.FluorophoreSystem(
    fluorophores=[custom_fluorophore],
)
transitions = fluorophore_system.load_transitions(
    wavelength=640,
    energy_transfer=False,
    dstorm=False,
)
transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Add a custom single-fluorophore transition

A custom photophysical process can introduce a new SingleState and one or more
TransitionTypes.

A complete state model should normally define both how the new state is entered
and how it is left. Each custom SingleState must have a name and numerical value
that do not conflict with another state in the TransitionSet.

To add the process:

1. Define the new SingleState.
2. Define the TransitionTypes entering and leaving that state.
3. Create Transitions containing their rates and fluorophore identities.
4. Add the Transitions under the corresponding fluorophore name.
5. Create a TransitionSet.

In [ ]:
import fluopy

custom_data = fluopy.FluorophoreData()
custom_fluorophore = fluopy.Fluorophore(
    name="custom",
    position=[0, 0],
    constants=custom_data,
)
fluorophore_system = fluopy.FluorophoreSystem(
    fluorophores=[custom_fluorophore],
)
# defining a new SingleState
dark = fluopy.SingleState(name="DARK", value=10)

# defining TransitionTypes for the new SingleState
dark_formation = fluopy.TransitionType(
    abbreviation="DARK_FORM",
    initial_state=fluopy.SingleState.S1,
    final_state=dark,
    photon=False,
)
recovery = fluopy.TransitionType(
    abbreviation="REC",
    initial_state=dark,
    final_state=fluopy.SingleState.S0,
    photon=False,
)

transitions = {
    "custom": [
        fluopy.Transition(
            transition_type=fluopy.TransitionType.EXCITATION,
            rate=1e6,
            fluorophore_ids=[0],
        ),
        fluopy.Transition(
            transition_type=fluopy.TransitionType.FLUORESCENT_EMISSION,
            rate=1e8,
            fluorophore_ids=[0],
        ),
        fluopy.Transition(
            transition_type=dark_formation,
            rate=1e4,
            fluorophore_ids=[0],
        ),
        fluopy.Transition(
            transition_type=recovery,
            rate=1e3,
            fluorophore_ids=[0],
        ),
    ]
}

transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Add a custom transition involving two fluorophores

A transition that changes two fluorophores simultaneously uses PairedState
objects. Each PairedState contains the donor state followed by the acceptor
state.

To define a paired transition:

1. Create the donor and acceptor fluorophores and their FluorophoreSystem.
2. Define any additional SingleStates.
3. Define the initial and final PairedStates.
4. Create a TransitionType from those paired states.
5. Create a Transition using donor–acceptor identity tuples.
6. Store it under a key containing the donor name, acceptor name, and distance.
7. Create a TransitionSet.

The SingleState components of paired transitions are included in the state space
of their corresponding fluorophores.

In [ ]:
import fluopy

donor = fluopy.Fluorophore(
    name="donor",
    position=[0, 0],
    constants=fluopy.FluorophoreData(),
)
acceptor = fluopy.Fluorophore(
    name="acceptor",
    position=[5, 0],
    constants=fluopy.FluorophoreData(),
)
fluorophore_system = fluopy.FluorophoreSystem([donor, acceptor])

# defining additional SingleStates
dark = fluopy.SingleState(name="DARK", value=10)
# defining initial and final PairedStates
initial_pair = fluopy.PairedState(
    name="S1_S0_CUSTOM",
    donor=fluopy.SingleState.S1,
    acceptor=fluopy.SingleState.S0,
)
final_pair = fluopy.PairedState(
    name="S0_DARK",
    donor=fluopy.SingleState.S0,
    acceptor=dark,
)
# creating a PairedState TransitionType
dark_transfer = fluopy.TransitionType(
    abbreviation="DARK_ET",
    initial_state=initial_pair,
    final_state=final_pair,
    photon=False,
)
# creating a SingleState TransitionType
dark_recovery = fluopy.TransitionType(
    abbreviation="DARK_REC",
    initial_state=dark,
    final_state=fluopy.SingleState.S0,
    photon=False,
)

donor_id = donor.identity
acceptor_id = acceptor.identity
distance = fluorophore_system.distances[(donor_id, acceptor_id)]
# store the PairedState transition using a key in the following format:
transition_key = f"D: {donor.name}, A: {acceptor.name}, dist: {distance}"

transitions = {
    donor.name: [
        fluopy.Transition(
            fluopy.TransitionType.EXCITATION,
            rate=1e6,
            fluorophore_ids=[donor_id],
        ),
        fluopy.Transition(
            fluopy.TransitionType.FLUORESCENT_EMISSION,
            rate=1e8,
            fluorophore_ids=[donor_id],
        ),
    ],
    acceptor.name: [
        fluopy.Transition(
            fluopy.TransitionType.EXCITATION,
            rate=1e6,
            fluorophore_ids=[acceptor_id],
        ),
        fluopy.Transition(
            fluopy.TransitionType.FLUORESCENT_EMISSION,
            rate=1e8,
            fluorophore_ids=[acceptor_id],
        ),
        fluopy.Transition(
            dark_recovery,
            rate=1e3,
            fluorophore_ids=[acceptor_id],
        ),
    ],
    # donor and acceptor identitiy tuples for the PairedState transition
    transition_key: [
        fluopy.Transition(
            dark_transfer,
            rate=1e6,
            fluorophore_ids=[(donor_id, acceptor_id)],
        )
    ],
}

transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Derive standard FRET automatically from custom spectra

FluorophoreSystem.load_transitions() can derive standard FRET transitions from
manually supplied spectra.

For each donor–acceptor pair, Fluopy:

1. Finds the common wavelength range of the donor emission and acceptor absorption.
2. Interpolates both spectra onto a shared wavelength grid.
3. Calculates the spectral overlap integral.
4. Derives the donor emission rate from its quantum yield and fluorescence lifetime.
5. Calculates the distance-dependent energy-transfer rate.
6. Creates the corresponding built-in energy-transfer transitions.

The fluorophore positions determine the donor–acceptor distance.
dipole_orientation_factor and refractive_index can be supplied through
energy_transfer_parameters.

In [ ]:
import fluopy

wavelengths = [500, 525, 550, 575, 600]
donor_emission = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[0.0, 0.4, 1.0, 0.6, 0.1],
)
donor_absorption = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[80000, 60000, 30000, 10000, 1000],
)
acceptor_emission = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[0.0, 0.1, 0.5, 1.0, 0.6],
)
acceptor_absorption = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[1000, 10000, 60000, 90000, 50000],
)

donor_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=donor_emission,
    absorption_spectra={"s0": donor_absorption},
)

acceptor_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.5,
    FLUORESCENCE_LIFETIME=4e-9,
    emission_spectrum=acceptor_emission,
    absorption_spectra={"s0": acceptor_absorption},
)

donor = fluopy.Fluorophore(
    name="donor",
    position=[0, 0],
    constants=donor_data,
)
acceptor = fluopy.Fluorophore(
    name="acceptor",
    position=[5, 0],
    constants=acceptor_data,
)

fluorophore_system = fluopy.FluorophoreSystem([donor, acceptor])
# load_transitions works with custom spectra
transitions = fluorophore_system.load_transitions(
    wavelength=550,
    energy_transfer=True,
    dstorm=False,
    energy_transfer_parameters={
        "dipole_orientation_factor": 2 / 3,
        "refractive_index": 1.33,
    },
)
transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Add a custom outcome to a supported energy-transfer channel

The include parameter adds a custom TransitionType to an existing acceptor
absorption channel. Supported channels are s0, s1, t1, cis, and off.

Each supplied factor determines the fraction of the calculated transfer rate
assigned to the custom transition. The remaining fraction is assigned to the
built-in transitions for that channel.

For example, a factor of 0.2 for an s0 channel produces:

- The custom transition with 20% of the calculated rate.
- Standard FRET with the remaining 80%.

A factor of 1 assigns the complete rate to the custom transition. include does
not automatically add later relaxation transitions required by the custom final
state.

In [1]:
import fluopy

wavelengths = [500, 525, 550, 575, 600]
donor_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=fluopy.Spectrum(
        wavelengths=wavelengths,
        values=[0.0, 0.4, 1.0, 0.6, 0.1],
    ),
    absorption_spectra={
        "s0": fluopy.Spectrum(
            wavelengths=wavelengths,
            values=[80000, 60000, 30000, 10000, 1000],
        )
    },
)
acceptor_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.5,
    FLUORESCENCE_LIFETIME=4e-9,
    ISC_TS_RATE=1e6,
    emission_spectrum=fluopy.Spectrum(
        wavelengths=wavelengths,
        values=[0.0, 0.1, 0.5, 1.0, 0.6],
    ),
    absorption_spectra={
        "s0": fluopy.Spectrum(
            wavelengths=wavelengths,
            values=[1000, 10000, 60000, 90000, 50000],
        )
    },
)

donor = fluopy.Fluorophore(name="donor", position=[0, 0], constants=donor_data)
acceptor = fluopy.Fluorophore(name="acceptor", position=[5, 0], constants=acceptor_data)
fluorophore_system = fluopy.FluorophoreSystem([donor, acceptor])

# this energy transfer is not automatically derived in load_transitions
triplet_transfer = fluopy.TransitionType(
    abbreviation="TET",
    initial_state=fluopy.PairedState.S1_S0,
    final_state=fluopy.PairedState.S0_T1,
    photon=False,
)
# but with the include parameter, it can still be derived directly, here with a 20%
# efficiency
transitions = fluorophore_system.load_transitions(
    wavelength=550,
    energy_transfer=True,
    dstorm=False,
    energy_transfer_parameters={
        "dipole_orientation_factor": 2 / 3,
        "refractive_index": 1.33,
        "include": {
            "s0": [(triplet_transfer, 0.2)],
        },
    },
)

transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

'overwrite', 'exclude' or 'include' in energy_transfer_parameters will effect all types of fluorophores.


## Define energy transfer involving a custom acceptor state

derive_energy_transfer_transitions() recognizes only the built-in absorption
channels s0, s1, t1, cis, and off.

For another acceptor state, first calculate the rate directly with
fluopy.transitions.derive_energy_transfer_rate(). Then create a custom
TransitionType and Transition using that rate.

The acceptor absorption spectrum describes absorption from the custom initial
acceptor state. The initial and final PairedStates determine how donor and
acceptor states change during the transfer.

In [ ]:
import fluopy

wavelengths = [500, 525, 550, 575, 600]
donor_emission = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[0.0, 0.4, 1.0, 0.6, 0.1],
)
donor_absorption = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[80000, 60000, 30000, 10000, 1000],
)
acceptor_emission = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[0.0, 0.1, 0.5, 1.0, 0.6],
)
acceptor_s0_absorption = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[1000, 10000, 60000, 90000, 50000],
)
dark_absorption = fluopy.Spectrum(
    wavelengths=wavelengths,
    values=[5000, 20000, 70000, 40000, 5000],
)

donor_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=donor_emission,
    absorption_spectra={"s0": donor_absorption},
)
acceptor_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.5,
    FLUORESCENCE_LIFETIME=4e-9,
    emission_spectrum=acceptor_emission,
    absorption_spectra={
        "s0": acceptor_s0_absorption,
        "dark": dark_absorption,
    },
)

donor = fluopy.Fluorophore(name="donor", position=[0, 0], constants=donor_data)
acceptor = fluopy.Fluorophore(name="acceptor", position=[5, 0], constants=acceptor_data)
fluorophore_system = fluopy.FluorophoreSystem([donor, acceptor])

dark = fluopy.SingleState(name="DARK", value=10)
dark_excited = fluopy.SingleState(name="DARK_EXCITED", value=11)
dark_formation = fluopy.TransitionType(
    abbreviation="DARK_FORM",
    initial_state=fluopy.SingleState.S0,
    final_state=dark,
    photon=False,
)
dark_relaxation = fluopy.TransitionType(
    abbreviation="DARK_RELAX",
    initial_state=dark_excited,
    final_state=fluopy.SingleState.S0,
    photon=False,
)
initial_pair = fluopy.PairedState(
    name="S1_DARK",
    donor=fluopy.SingleState.S1,
    acceptor=dark,
)
final_pair = fluopy.PairedState(
    name="S0_DARK_EXCITED",
    donor=fluopy.SingleState.S0,
    acceptor=dark_excited,
)
custom_et = fluopy.TransitionType(
    abbreviation="DARK_ET",
    initial_state=initial_pair,
    final_state=final_pair,
    photon=False,
)

transitions = fluorophore_system.load_transitions(
    wavelength=550,
    energy_transfer=False,
    dstorm=False,
)
transitions[acceptor.name].extend(
    [
        fluopy.Transition(
            dark_formation, rate=1e3, fluorophore_ids=[acceptor.identity]
        ),
        fluopy.Transition(
            dark_relaxation, rate=1e4, fluorophore_ids=[acceptor.identity]
        ),
    ]
)

pair = (donor.identity, acceptor.identity)
distance = fluorophore_system.distances[pair]
# the rate can be derived depending on the spectrum data
rate = fluopy.transitions.derive_energy_transfer_rate(
    donor_data=donor.constants,
    acceptor_absorption=dark_absorption,
    distance=distance,
)

transition_key = f"D: {donor.name}, A: {acceptor.name}, dist: {distance}"
transitions.setdefault(transition_key, []).append(
    fluopy.Transition(
        transition_type=custom_et,
        rate=rate,
        fluorophore_ids=[pair],
    )
)
transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Input requirements

Spectrum wavelengths are given in nm and must be strictly increasing. Spectrum 
intensities must be non-negative.

Absorption spectra used for excitation or energy-transfer calculations must
contain absolute molar extinction coefficients in 1/(M cm). Emission spectra may
contain relative intensities because Fluopy normalizes them for spectral-overlap
calculations.

Energy-transfer distances are given in nm. Fluorescence lifetimes are given in
seconds, and transition rates are given in 1/s.

Individual absorption cross sections apply only at their specified excitation
wavelength. Energy-transfer calculations use complete absorption spectra rather
than individual cross sections.